In [74]:
from tqdm import tqdm
import pandas as pd
import os
import ast
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import mlflow
import mlflow.pytorch
from transformers import pipeline

mlflow.set_tracking_uri("http://localhost:8080")

In [75]:
mimic = pd.read_csv("../data/mimic.csv")
motu = pd.read_csv("../data/motu.csv")
sepsi = pd.read_csv("../data/sepsi.csv")

test = pd.concat([mimic, motu, sepsi])

In [ ]:
os.environ["HF_DATASETS_CACHE"] = "/mnt/data_ml/hf_cache/datasets"
os.environ["TMPDIR"] = "/mnt/data_ml/tmp"

In [76]:
test['Mapping'] = [ast.literal_eval(t) for t in test['Mapping']]
test = test[~test['Field_description'].isna()]
test = test.explode('Mapping')
test = test.loc[:, ['Field_description', 'Mapping']].rename(columns={'Field_description': 'input', 'Mapping':'output'})

sphn = pd.read_csv("../data/sphn.csv")
sphn.rename(columns={'Field_description':'input', 'Mapping':'output'}, inplace=True)

test = pd.concat([test, sphn])

test.to_json("output.jsonl", orient='records', lines=True, force_ascii=False)

In [79]:
df = load_dataset(
    "json",
    data_files="/mnt/data_ml/workspace/flex-llm-care/output.jsonl",
    split='train'
)

In [ ]:
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base", cache_dir="/mnt/data_ml/hf_cache")
model = AutoModelForSeq2SeqLM.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base", cache_dir="/mnt/data_ml/hf_cache", revision='main').to(device)

def paraphrase(
    question,
    num_beams=3,
    num_beam_groups=3,
    num_return_sequences=3,
    repetition_penalty=10.0,
    diversity_penalty=3.0,
    no_repeat_ngram_size=2,
    temperature=0.7,
    max_length=128
):
    input_ids = tokenizer(
        f'paraphrase: {question}',
        return_tensors="pt", padding="longest",
        max_length=max_length,
        truncation=True,
    ).input_ids.to(device)

    outputs = model.generate(
        input_ids, temperature=temperature, repetition_penalty=repetition_penalty,
        num_return_sequences=num_return_sequences, no_repeat_ngram_size=no_repeat_ngram_size,
        num_beams=num_beams, num_beam_groups=num_beam_groups,
        max_length=max_length, diversity_penalty=diversity_penalty,trust_remote_code=True
    )

    res = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    return res

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [42]:
mlflow.set_experiment("fhir_mistral_paraphrasing")

<Experiment: artifact_location='mlflow-artifacts:/249965773614738610', creation_time=1771503126483, experiment_id='249965773614738610', last_update_time=1771503126483, lifecycle_stage='active', name='fhir_mistral_paraphrasing', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [86]:
test['resources'] = [t.split(".")[0] for t in test['output']]
test['resources'].value_counts()

resources
Observation                 124
QuestionnaireResponse        98
Questionnaire                98
DiagnosticReport             47
Encounter                    37
Procedure                    33
MedicationAdministration     30
Patient                      19
MedicationRequest            18
Condition                    15
Device                       14
Specimen                     12
DeviceDefinition             12
MedicationDispense           10
Obeservation                  7
ServiceRequest                4
AllergyIntolerance            3
AdverseEvent                  2
Location                      2
Practioner                    2
Medication                    2
Allergy                       2
Organization                  1
Communication                 1
CarePlan                      1
Consent                       1
ChargeItem                    1
PractionerRole                1
Substance                     1
Name: count, dtype: int64

In [ ]:
with mlflow.start_run(run_name="data_preprocessing_and_paraphrase"):
    # Log dataset sizes
    mlflow.log_param("original_rows", len(test))

    new_rows = []

    for _, row in tqdm(test.iterrows(), total=len(test), desc="Paraphrasing"):
        outputs = paraphrase(row['input'])
        #print(outputs)
        for o in outputs:
            new_rows.append({'input': o, 'output': row['output']})

    mlflow.log_metric("num_paraphrases_generated", len(new_rows))

    # Save augmented dataset as an artifact
    augmented_df = pd.concat([test, pd.DataFrame(new_rows)])
    augmented_file = "output.jsonl"
    augmented_df.to_json(augmented_file, orient='records', lines=True, force_ascii=False)
    mlflow.log_artifact(augmented_file)

Paraphrasing:   0%|          | 0/531 [00:00<?, ?it/s]Group Beam Search was moved to a `custom_generate` repo: https://hf.co/transformers-community/group-beam-search. To prevent loss of backward compatibility, add `custom_generate='transformers-community/group-beam-search'` to your `generate` call before v4.62.0.
Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length', 'diversity_penalty', 'no_repeat_ngram_size', 'num_beams', 'repetition_penalty', 'temperature', 'num_beam_groups'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Paraphrasing: 100%|██████████| 531/531 [05:39<00:00,  1.57it/s]


🏃 View run data_preprocessing_and_paraphrase at: http://localhost:8080/#/experiments/249965773614738610/runs/664ccdd82ca24da8b20edc3032fd52ab
🧪 View experiment at: http://localhost:8080/#/experiments/249965773614738610


In [44]:
# Log number of augmented examples

prova = pd.concat([test, pd.DataFrame(new_rows)])
prova.to_json("output.jsonl", orient='records', lines=True, force_ascii=False)

In [45]:
# """ from sentence_transformers import SentenceTransformer, util

# # Carica modello embeddings
# model = SentenceTransformer('all-MiniLM-L6-v2')  # leggero e gratuito

# original = "Unique patient identifier"
# paraphrases = [
#     'An exclusive label for a particular patient.',
#     'A distinct patient designation.',
#     'Novel patient designation',
#     'Personalized diagnosis label for patient.',
#     'Unique identification number of patient'
# ]

# # Calcola embeddings
# emb_original = model.encode(original, convert_to_tensor=True)
# emb_paraphrases = model.encode(paraphrases, convert_to_tensor=True)

# # Calcola similarità coseno
# cos_scores = util.cos_sim(emb_original, emb_paraphrases)

# # Ordina per similarità
# sorted_idx = cos_scores[0].argsort(descending=True)
# best_paraphrases = [paraphrases[i] for i in sorted_idx]

# print("Parafrasi più vicine all'originale:")
# for p in best_paraphrases:
#     print(p) """

In [46]:
#export NLTK_DATA=/path/che/vuoi/nltk_data
#os.environ['NLTK_DATA'] = ''

In [47]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir="/mnt/data_ml/hf_cache", device_map='cuda')
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir="/mnt/data_ml/hf_cache", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [52]:
test['resources'] = [o.split(".")[0] for o in test['output']]
test['resources'].value_counts()

resources
Observation                 107
Questionnaire                98
QuestionnaireResponse        98
DiagnosticReport             47
Encounter                    35
MedicationAdministration     30
Procedure                    24
MedicationRequest            17
DeviceDefinition             12
Patient                      12
Device                       12
MedicationDispense           10
Specimen                      9
Condition                     8
ServiceRequest                4
AllergyIntolerance            3
Location                      2
AdverseEvent                  1
Communication                 1
CarePlan                      1
Name: count, dtype: int64

In [ ]:
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    optim="adamw_torch",
    save_steps=25,
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    packing=False,
)

peft_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

def formatting_func(example):
    return f"###Input: {example['input']} Response: {example['output']}"

trainer = SFTTrainer(
    model=model,
    train_dataset=df,
    peft_config=peft_config,
    formatting_func=formatting_func,
    args=sft_config,
)

with mlflow.start_run(run_name="sft_training"):
    # Log hyperparameters
    mlflow.log_params({
        "num_train_epochs": sft_config.num_train_epochs,
        "batch_size": sft_config.per_device_train_batch_size,
        "gradient_accumulation_steps": sft_config.gradient_accumulation_steps,
        "learning_rate": sft_config.learning_rate,
        "weight_decay": sft_config.weight_decay,
        "fp16": sft_config.fp16,
        "max_grad_norm": sft_config.max_grad_norm,
        "warmup_ratio": sft_config.warmup_ratio
    })

    trainer.train()

    new_model = "../models/fhir-mistral"
    model_name = os.path.basename(new_model)
    trainer.save_model(new_model)
    mlflow.log_artifacts(new_model, artifact_path=model_name)

    for step, loss in enumerate(trainer.state.log_history, 1):
        if 'loss' in loss:
            mlflow.log_metric("training_loss", loss['loss'], step=step)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,5.236396
2,5.114337
3,5.036678


🏃 View run sft_training at: http://localhost:8080/#/experiments/249965773614738610/runs/e436634228044a238636033215577863
🧪 View experiment at: http://localhost:8080/#/experiments/249965773614738610


KeyboardInterrupt: 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

adapter_path = "../models/fhir-mistral"

# applica LoRA
model = PeftModel.from_pretrained(
    model,
    adapter_path,
    local_files_only=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=200,
    do_sample=False
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
sphn = pd.read_csv("../data/sphn.csv")

results = []

for question in sphn['Field_description']:
    test = f"###Input: {question} Response: "
    result = pipe(test)
    results.append(result[0]['generated_text'].replace(test,"").strip())

In [ ]:
count = 0

for t in zip(sphn['Mapping'], results):
    values = [v.split(".")[0] for v in t]
    if values[0] == values[1]:
        count += 1

print(count / len(sphn['Mapping']))

0.11940298507462686
